# 🛡️ Email & SMS Spam Classification Using Machine Learning
### A Complete, End-to-End, Beginner-Friendly Machine Learning Guide

---

## 📌 Project Overview
This project builds an end-to-end Machine Learning pipeline that classifies incoming messages as:
- **Spam**: Unwanted, fraudulent, phishing, or promotional messages.
- **Ham**: Legitimate, safe, non-spam personal or professional communications.

### 🎯 Learning Objectives:
1. **Dataset Collection & Loading**: Fetch and verify the official UCI SMS Spam Collection dataset.
2. **Exploratory Data Analysis (EDA)**: Understand class imbalance, text lengths, and vocabulary differences.
3. **Text Preprocessing**: Tokenization, punctuation removal, stopword filtering, and Porter stemming.
4. **Feature Extraction**: TF-IDF (Term Frequency - Inverse Document Frequency) with n-grams.
5. **Model Training & Comparison**: Benchmark **Multinomial Naive Bayes**, **Logistic Regression**, **Support Vector Machines (SVC)**, and **Random Forest**.
6. **Evaluation Metrics**: Understand why **Precision** is far more critical than raw Accuracy in spam detection.
7. **Hyperparameter Tuning**: Optimize smoothing parameters via 5-fold cross-validation.
8. **Pipeline Export & Inference**: Save a production pipeline with `joblib` and classify novel emails in real time.

---

### 🚀 Running on Google Colab
If you are opening this notebook in Google Colab, run the following command in the first cell:
```python
!pip install -q scikit-learn pandas numpy matplotlib seaborn nltk joblib
```


In [ ]:
# Step 0: Import Standard Libraries
import os
import re
import zipfile
import urllib.request
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Ensure NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Visual style
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded successfully!')

## 1. Dataset Collection and Loading

We use the **UCI SMS Spam Collection** corpus.
- **Official UCI Repository**: [https://archive.ics.uci.edu/dataset/228/sms+spam+collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)
- **Direct Archive**: `https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip`

> **Note on Dataset Scope:** This dataset contains SMS messages rather than multi-paragraph emails. However, the exact same text normalization, TF-IDF vectorization, and Bayesian/SVM classification pipeline applies directly to email text bodies!

In [ ]:
# Download and verify dataset
data_dir = '../data/raw'
os.makedirs(data_dir, exist_ok=True)
zip_path = os.path.join(data_dir, 'sms_spam_collection.zip')
raw_file_path = os.path.join(data_dir, 'SMSSpamCollection')

if not os.path.exists(raw_file_path):
    url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
    print('Downloading dataset from UCI repository...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print('Extraction complete!')
else:
    print('Dataset already available at:', raw_file_path)

# Load into DataFrame (tab-separated: label \t message)
df = pd.read_csv(raw_file_path, sep='\t', header=None, names=['label', 'message'], quoting=3)
print(f'Initial Raw Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

Let's audit:
1. Missing values
2. Duplicate records (and why removing duplicates is essential to prevent train-test data leakage)
3. Class distribution (Ham vs Spam)
4. Character length & word count distributions

In [ ]:
# Check Missing Values & Duplicates
print('Missing Values per Column:\n', df.isnull().sum())
print(f'Total Duplicate Messages: {df.duplicated(subset=["message"]).sum()}')

# Deduplicate messages
# Retaining duplicates allows identical messages in both train and test splits, artificially boosting test metrics.
df = df.drop_duplicates(subset=['message']).copy().reset_index(drop=True)
print(f'Clean Deduplicated Shape: {df.shape}')

# Class Distribution
ham_count = (df['label'] == 'ham').sum()
spam_count = (df['label'] == 'spam').sum()
print(f'Ham  (Legitimate): {ham_count:,} ({ham_count / len(df) * 100:.2f}%)')
print(f'Spam (Unwanted)  : {spam_count:,} ({spam_count / len(df) * 100:.2f}%)')
print(f'Class Imbalance Ratio: ~{ham_count / spam_count:.1f}:1')

# Add length metrics
df['char_count'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(str(x).split()))
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

In [ ]:
# Visualize Class Imbalance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['#2b5c8f', '#d9534f']

sns.barplot(x=['Ham', 'Spam'], y=[ham_count, spam_count], palette=colors, ax=ax1, edgecolor='black', width=0.5)
ax1.set_title('Message Frequency by Class', fontweight='bold')
ax1.set_ylabel('Total Count')

ax2.pie([ham_count, spam_count], labels=['Ham', 'Spam'], autopct='%1.1f%%', colors=colors, explode=(0, 0.08), shadow=True)
ax2.set_title('Class Proportions (~87% Ham vs ~13% Spam)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare Message Lengths (Character Count & Word Count)
print('Descriptive Text Statistics:')
print(df.groupby('label')[['char_count', 'word_count']].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Character count density
sns.histplot(df[df['label'] == 'ham']['char_count'], color='#2b5c8f', label='Ham (Mean ~71 chars)', kde=True, bins=40, stat='density', ax=ax1)
sns.histplot(df[df['label'] == 'spam']['char_count'], color='#d9534f', label='Spam (Mean ~138 chars)', kde=True, bins=40, stat='density', ax=ax1)
ax1.set_xlim(0, 300)
ax1.set_title('Character Length: Ham vs Spam', fontweight='bold')
ax1.set_xlabel('Character Count')
ax1.legend()

# Word count boxplot
sns.boxplot(x='label', y='word_count', hue='label', legend=False, data=df, palette=colors, ax=ax2, width=0.4, showmeans=True)
ax2.set_title('Word Count Distribution (Mean = Yellow dot)', fontweight='bold')
ax2.set_ylim(0, 50)
plt.tight_layout()
plt.show()

### 💡 Key EDA Findings:
1. **Imbalance**: Ham accounts for **87.37%** and Spam accounts for **12.63%**. A dummy model guessing Ham every time would achieve 87.37% accuracy, proving that **Accuracy is an inadequate standalone metric**.
2. **Length Disparity**: Spam messages average **137.7 characters** (tightly clustered around the 160-char SMS carrier limit), whereas Ham messages average **70.9 characters** with wide variance.

## 3. Text Cleaning and Preprocessing

Raw text contains noise that distracts machine learning algorithms: casing differences, URLs, email addresses, punctuation, numbers, and stopwords.

We build a robust preprocessing function:
1. **Lowercasing**: Unifies 'FREE', 'Free', and 'free'.
2. **Regex Cleaning**: Strips web URLs (`http...`), email addresses, and punctuation.
3. **Tokenization**: Splits text into individual words.
4. **Stopword Removal**: Eliminates non-informative grammatical words ('the', 'is', 'in').
5. **Porter Stemming**: Reduces inflected words to root form ('winnings', 'winner' -> 'win').

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs & Emails
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text)
    # 3. Strip non-alphabet characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # 4. Tokenize & remove stopwords
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 1]
    # 5. Porter Stemming
    stemmed_tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(stemmed_tokens)

# Demonstrate preprocessing on a sample spam message
sample_raw = df[df['label'] == 'spam']['message'].iloc[0]
print('--- Before Preprocessing ---')
print(sample_raw)
print('\n--- After Preprocessing ---')
print(clean_text(sample_raw))

# Apply to entire dataset
df['cleaned_message'] = df['message'].apply(clean_text)
# Fallback for messages that became empty
empty_mask = df['cleaned_message'].str.strip() == ''
df.loc[empty_mask, 'cleaned_message'] = df.loc[empty_mask, 'message'].str.lower()
print(f'Preprocessing complete on {len(df)} records!')

## 4. Train/Test Split and TF-IDF Feature Extraction

### Why Stratification is Essential
Because our dataset has an 87:13 class ratio, standard random sampling might under-sample or over-sample spam in the test partition. **Stratified Splitting** (`stratify=y`) guarantees that both training (80%) and testing (20%) sets preserve the exact 87.37% Ham / 12.63% Spam proportion.

### Understanding TF-IDF
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$
- **TF (Term Frequency)**: Frequency of word $t$ in message $d$.
- **IDF (Inverse Document Frequency)**: Logarithmic penalty for words appearing across all messages.
We also include **bigrams** (`ngram_range=(1, 2)`) to capture phrases like 'call now' or 'free gift'.

In [ ]:
X = df['cleaned_message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Set: {len(X_train)} samples (Ham: {(y_train == 0).sum()}, Spam: {(y_train == 1).sum()})')
print(f'Testing Set : {len(X_test)} samples (Ham: {(y_test == 0).sum()}, Spam: {(y_test == 1).sum()})')

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(f'TF-IDF Feature Matrix Shape: {X_train_vec.shape}')
print(f'Sample Feature Names: {tfidf.get_feature_names_out()[:15]}')

## 5. Model Training & Benchmarking

We train and benchmark four distinct machine learning algorithms:
1. **Multinomial Naive Bayes (MNB)**: Probabilistic model applying Bayes' theorem with feature independence.
2. **Logistic Regression (LR)**: Linear classifier with sigmoid function optimizing log-loss.
3. **Support Vector Machine (Linear SVC)**: Finds the maximum-margin hyperplane separating classes.
4. **Random Forest (RF)**: Ensemble of 100 decision trees with bootstrap aggregation.

> **Interview Note:** In spam detection, **Precision is the most critical metric**. A False Positive sends a legitimate email (e.g., job offer, flight ticket) to the Spam folder, which is catastrophic. A False Negative merely leaves an unwanted ad in the Inbox.

In [ ]:
classifiers = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Linear SVM': SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

benchmark_results = {}

for name, clf in classifiers.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    roc = roc_auc_score(y_test, clf.predict_proba(X_test_vec)[:, 1])
    cm = confusion_matrix(y_test, y_pred)
    
    benchmark_results[name] = {
        'Accuracy': acc,
        'Precision (Spam)': prec,
        'Recall (Spam)': rec,
        'F1-Score': f1,
        'ROC-AUC': roc,
        'False Positives': cm[0][1],
        'False Negatives': cm[1][0]
    }

results_df = pd.DataFrame(benchmark_results).T
results_df.style.format('{:.2%}', subset=['Accuracy', 'Precision (Spam)', 'Recall (Spam)', 'F1-Score', 'ROC-AUC'])

In [ ]:
# Benchmark Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ['Accuracy', 'Precision (Spam)', 'Recall (Spam)', 'F1-Score']
results_df[metrics_to_plot].plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black', width=0.75)
ax.set_title('Model Performance Benchmark on Test Set', fontweight='bold')
ax.set_ylabel('Score (0 - 1.0)')
ax.set_ylim(0.7, 1.05)
ax.set_xticklabels(results_df.index, rotation=0, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning (GridSearchCV)

In **Multinomial Naive Bayes**, the Laplace smoothing parameter $\alpha$ controls how aggressively we smooth unseen word frequencies.
- Default $\alpha=1.0$ (Laplace smoothing).
- Smaller $\alpha$ (Lidstone smoothing) assigns more weight to rare tokens, increasing recall but potentially risking false positives.
We tune $\alpha$ across `[0.05, 0.1, 0.2, 0.5, 1.0]` with 5-fold cross-validation.

In [ ]:
param_grid = {'alpha': [0.05, 0.1, 0.2, 0.5, 1.0]}
grid = GridSearchCV(MultinomialNB(), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train_vec, y_train)

print(f'Best alpha parameter: {grid.best_params_}')
print(f'Best 5-Fold Cross-Validation F1-Score: {grid.best_score_ * 100:.2f}%')

tuned_model = grid.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_vec)
print('\nTuned Model Classification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=['Ham (0)', 'Spam (1)'], digits=4))

## 7. Confusion Matrix and Error Analysis

Let's visualize the confusion matrix and inspect exactly what messages resulted in errors.

In [ ]:
cm = confusion_matrix(y_test, classifiers['Multinomial Naive Bayes'].predict(X_test_vec))
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Predicted Ham', 'Predicted Spam'],
            yticklabels=['Actual Ham', 'Actual Spam'])
ax.set_title('Confusion Matrix: Multinomial Naive Bayes (alpha=1.0)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True Negatives (Legitimate Ham Correct): {tn}')
print(f'False Positives (Ham Flagged as Spam) : {fp}  <-- Only 1 false alarm!')
print(f'False Negatives (Missed Spam)           : {fn}')
print(f'True Positives  (Spam Caught)          : {tp}')

## 8. Exporting the Production Pipeline and Real-Time Inference

A clean production deployment bundles the **TF-IDF Vectorizer** and the **trained Classifier** inside a single `scikit-learn Pipeline`. This prevents train/serving skew and makes inference a single function call!

In [ ]:
# Create end-to-end Pipeline
production_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)),
    ('classifier', MultinomialNB(alpha=1.0))
])

# Fit pipeline on training data
production_pipeline.fit(X_train, y_train)

# Save pipeline to disk
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, 'spam_classifier_pipeline.joblib')
joblib.dump(production_pipeline, model_path)
print(f'Pipeline saved successfully to: {model_path}')

# Real-Time Inference Function
def predict_new_email(text: str):
    cleaned = clean_text(text)
    pred = production_pipeline.predict([cleaned])[0]
    probs = production_pipeline.predict_proba([cleaned])[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    confidence = probs[pred] * 100
    
    print(f'Message   : "{text}"')
    print(f'Prediction: [{label}] (Confidence: {confidence:.2f}%)')
    print(f'Ham Prob  : {probs[0]*100:.2f}% | Spam Prob: {probs[1]*100:.2f}%\n')

# Test realistic novel samples
predict_new_email('Congratulations! You have won a 5,000 dollar shopping voucher. Click here to claim your reward now.')
predict_new_email('Hi Professor, I have pushed the assignment code to GitHub. Please review it when you have time.')
predict_new_email('Urgent: Your account password has expired. Text RESET to 89899 to keep access.')
predict_new_email('Hey mom, what time should we meet at the grocery store today?')

## 9. 🧪 Interactive Testing: Test Your Own Custom Email

Run the cell below and type or paste **any email, message, or subject line** into the prompt box to see the prediction in real time!

In [ ]:
# Type or paste your own message here to test:
user_test_msg = input("Enter email / SMS text to classify: ")
if user_test_msg.strip():
    predict_new_email(user_test_msg)
else:
    print("Please enter a message to test.")


## 10. Conclusion & Summary

### Summary of Accomplishments:
- Loaded, audited, and cleaned the **UCI SMS Spam Collection** corpus.
- Handled missing values, removed duplicate messages to eliminate train/test leakage, and quantified class imbalance.
- Built NLP preprocessing functions including tokenization, lowercasing, regex filtering, stopword removal, and Porter stemming.
- Trained and compared four supervised machine learning models using stratified splitting and TF-IDF feature extraction.
- Established that **Multinomial Naive Bayes** delivers exceptional **99.04% Precision**, achieving only 1 False Positive across 1,035 test instances.
- Serialized the final model into a reusable pipeline ready for deployment via CLI or Streamlit Web App (`app.py`).